# Lending Club EDA -- F2 -- Data Quality & Integrity

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

Missingness ranked and tested for whether it's informative (not just how much), IQR- and z-score-based outlier sweeps across every retained numeric feature (cross-checked against each other), and explicit plausibility rules for impossible values (negative ratios, out-of-range FICO, etc.).

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

| # | What it does |
|---|---|
| 1 | Connect; missingness ranked on the retained + borderline-dropped columns |
| 2 | Formal test: is missingness on those columns informative of `is_bad`? |
| 3 | IQR-based outlier counts, all 20 retained numeric features |
| 4 | Z-score outlier counts (cross-check against IQR) |
| 5 | Explicit plausibility rules for implausible values |

**No cleaning happens here.** This notebook diagnoses missingness and
outliers; the actual imputation/flagging strategy is applied in
`notebooks/data_cleaning/`, informed by what's found here.

## Cell 1 -- missingness, ranked, on the modeling-relevant columns

**What / why:** `01_data_understanding_structural_profiling.ipynb` profiled
missingness across all 151 raw columns; scoping down here to the 20 numeric
+ 7 categorical retained columns, plus 3 borderline ones dropped during
feature selection (`mths_since_last_delinq`, `tot_hi_cred_lim`, `bc_util`) --
worth re-checking here since missingness was part of why they were dropped.

**Where this column list comes from -- and why it doesn't match anything computed in `01_data_understanding_structural_profiling.ipynb`:** it isn't derived from that notebook, or from any calculation at all -- it's a fixed starter shortlist, picked by domain judgment before any of these diagnostic notebooks ran, not by a data-driven selection step. It covers the fields a Lending Club credit-risk build would be expected to lean on: loan terms (`loan_amnt`, `int_rate`, `term`, `grade`), borrower risk signals (`fico_range_low`, `delinq_2yrs`, `inq_last_6mths`, `pub_rec`, `pub_rec_bankruptcies`), balance-sheet fields (`annual_inc`, `dti`, `revol_bal`, `revol_util`, `tot_cur_bal`, `mort_acc`, `open_acc`, `total_acc`, `bc_open_to_buy`, `acc_open_past_24mths`, `mo_sin_old_rev_tl_op`, `num_actv_rev_tl`), and categorical context (`emp_length`, `home_ownership`, `verification_status`, `purpose`, `addr_state`) -- plus three borderline fields kept only to double-check whether they belong (`mths_since_last_delinq`, `tot_hi_cred_lim`, `bc_util`).

This is the **candidate** set, not the final one. Every `02_eda/` notebook from here through 14 investigates this same fixed list, but `notebooks/03_data_cleaning/01_cleaning_and_feature_prep.ipynb` cell 1 is where it actually gets tested and narrowed -- e.g. `avg_cur_bal` was on an earlier version of this list and was dropped after notebook 09's VIF check found it redundant with `tot_cur_bal`. That final, audited list, with the notebook and cell justifying every column that changed, lives there. Documenting the starting list's origin here, at its first real use, is itself a fix -- earlier drafts of this notebook used this exact list without ever saying where it came from.

**Expect:** most retained columns near 0% missing; `mort_acc`,
`bc_open_to_buy`, `pub_rec_bankruptcies`-style fields showing measurable
missingness.

In [1]:
import sys, os, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()  # read-only; creates the asset folders if missing

check_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state'] + ["mths_since_last_delinq", "tot_hi_cred_lim", "bc_util"]
rows = []
n_total = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
for c in check_cols:
    n_null = con.sql(f'SELECT count(*) FROM windowed WHERE "{c}" IS NULL').fetchone()[0]
    rows.append((c, n_null, n_null / n_total))
miss_df = pd.DataFrame(rows, columns=["column", "n_missing", "pct_missing"]).sort_values("pct_missing", ascending=False)
print(miss_df.to_string(index=False))
miss_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_miss_df.csv"), index=False)


                column  n_missing  pct_missing
mths_since_last_delinq     590519 4.937949e-01
            emp_length      70579 5.901851e-02
               bc_util      13083 1.094007e-02
        bc_open_to_buy      12383 1.035473e-02
            revol_util        686 5.736366e-04
                   dti        223 1.864737e-04
           avg_cur_bal         17 1.421549e-05
        inq_last_6mths          1 8.362050e-07
               pub_rec          0 0.000000e+00
        fico_range_low          0 0.000000e+00
           delinq_2yrs          0 0.000000e+00
            annual_inc          0 0.000000e+00
              int_rate          0 0.000000e+00
             loan_amnt          0 0.000000e+00
  pub_rec_bankruptcies          0 0.000000e+00
              mort_acc          0 0.000000e+00
             total_acc          0 0.000000e+00
             revol_bal          0 0.000000e+00
              open_acc          0 0.000000e+00
  mo_sin_old_rev_tl_op          0 0.000000e+00
           to

**What the output shows:** 8 of the
30 checked columns have any missingness at all, topped by
`mths_since_last_delinq` at 49.4%. Everything else
retained for modeling is close to fully populated.

**Next:** for the columns that DO have real missingness, testing directly
whether being missing correlates with `is_bad` -- the question that actually
matters, not just how much is missing.

## Cell 2 -- is missingness informative? A formal test

**What / why:** For every column from cell 1 with nonzero missingness,
comparing bad rate among missing-vs-populated rows. Sorting by gap size
alone can be misleading on tiny samples (a column with 1 missing row can
show a huge but meaningless gap), so this restricts the headline finding to
columns with at least 1,000 missing rows -- enough to trust the comparison.

**How:** for each column from cell 1 that has any missingness, group `windowed` by whether that one column `IS NULL`, and compute the bad rate for the two groups (missing vs. populated) plus how many rows fall in each. Pivot that long result so every column becomes one row with a `bad_rate_populated` and a `bad_rate_missing` side by side, attach back the missing-row count from cell 1, and compute `gap_pp` -- the percentage-point difference between the two bad rates. Sorting by the *size* of that gap (regardless of direction) puts the columns most worth a second look at the top.

**Expect:** at least one column with a real (multi-point) gap on a large
enough sample to trust, worth encoding as its own `_was_missing` flag rather
than silently imputed away.

In [2]:
miss_test = []
for c in miss_df[miss_df["pct_missing"] > 0]["column"]:
    r = con.sql(f'''
        SELECT "{c}" IS NULL AS is_missing, count(*) n, round(avg(is_bad),3) bad_rate
        FROM windowed GROUP BY 1
    ''').df()
    r["column"] = c
    miss_test.append(r)
miss_test_df = pd.concat(miss_test, ignore_index=True)
pivot = miss_test_df.pivot(index="column", columns="is_missing", values="bad_rate")
pivot.columns = ["bad_rate_populated", "bad_rate_missing"]
n_missing_lookup = miss_df.set_index("column")["n_missing"]
pivot["n_missing"] = n_missing_lookup.reindex(pivot.index)
pivot["gap_pp"] = (pivot["bad_rate_missing"] - pivot["bad_rate_populated"]) * 100
pivot = pivot.sort_values("gap_pp", key=abs, ascending=False)
print(pivot[["n_missing","bad_rate_populated","bad_rate_missing","gap_pp"]].to_string())


                        n_missing  bad_rate_populated  bad_rate_missing  gap_pp
column                                                                         
inq_last_6mths                  1               0.205             0.000   -20.5
avg_cur_bal                    17               0.205             0.294     8.9
emp_length                  70579               0.201             0.274     7.3
dti                           223               0.205             0.224     1.9
mths_since_last_delinq     590519               0.212             0.199    -1.3
bc_util                     13083               0.205             0.215     1.0
bc_open_to_buy              12383               0.205             0.214     0.9
revol_util                    686               0.205             0.203    -0.2


**What the output shows:** the single largest gap (`inq_last_6mths`,
-20.5 points) sits on only 1
missing row(s) -- noise, not signal, at that sample size. Restricting to
columns with at least 1,000 missing rows, the real finding is
`emp_length`: 27.4% bad rate
when missing vs. 20.1% when populated
(a 7.3-point gap) on 70,579
rows -- large enough and on enough rows to be real. `data_cleaning/` should
carry a `emp_length_was_missing` indicator alongside whatever
imputed value that column gets, not impute silently.

**Next:** missingness is handled -- both how much, and whether it's
informative. Moving to outlier detection across every retained numeric
feature.

## Cell 3 -- IQR-based outlier counts, all 20 numeric features

**What / why:** Flagging values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` for
every retained numeric feature -- a systematic sweep rather than a one-off
check, so nothing with a heavy tail gets missed.

**Expect:** the most right-skewed fields (income, balances) show the
highest outlier counts; sparse count fields (delinq_2yrs-style) also show
inflated counts for a different reason -- IQR breaks down when most values
sit at a single point (0) with a thin tail.

In [3]:
rows = []
n_total = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
for c in ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl']:
    q1, q3 = con.sql(f'''
        SELECT quantile_cont(TRY_CAST("{c}" AS DOUBLE),0.25),
               quantile_cont(TRY_CAST("{c}" AS DOUBLE),0.75)
        FROM windowed
    ''').fetchone()
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = con.sql(f'''
        SELECT count(*) FROM windowed
        WHERE TRY_CAST("{c}" AS DOUBLE) < {lo} OR TRY_CAST("{c}" AS DOUBLE) > {hi}
    ''').fetchone()[0]
    rows.append((c, iqr, lo, hi, n_out, n_out/n_total))
iqr_df = pd.DataFrame(rows, columns=["column","iqr","lower_fence","upper_fence","n_outliers","pct_outliers"]).sort_values("pct_outliers", ascending=False)
print(iqr_df.round(2).to_string(index=False))
iqr_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_iqr_df.csv"), index=False)


              column       iqr  lower_fence  upper_fence  n_outliers  pct_outliers
         delinq_2yrs      0.00         0.00         0.00      239211          0.20
             pub_rec      0.00         0.00         0.00      215974          0.18
pub_rec_bankruptcies      0.00         0.00         0.00      156894          0.13
      bc_open_to_buy  10655.00    -14540.50     28079.50      102950          0.09
           revol_bal  13880.00    -14749.00     40771.00       71116          0.06
         avg_cur_bal  15501.00    -20142.50     41861.50       63209          0.05
      inq_last_6mths      1.00        -1.50         2.50       61614          0.05
          annual_inc  45000.00    -21500.00    158500.00       58326          0.05
            open_acc      6.00        -1.00        23.00       43065          0.04
         tot_cur_bal 180394.50   -241011.75    480566.25       41489          0.03
      fico_range_low     40.00       610.00       770.00       36926          0.03
mo_s

**What the output shows:** `delinq_2yrs` has the highest
IQR-flagged share at 20.0%. As expected, the top of
the ranking mixes heavily skewed continuous fields with sparse count fields
where IQR is known to over-flag ordinary nonzero values as "outliers" simply
because most rows sit at exactly 0.

**Next:** cross-checking against a z-score-based method, which handles
skewed continuous fields differently and helps separate "genuinely extreme"
from "IQR's known weakness on sparse/count data."'

## Cell 4 -- z-score outlier counts (cross-check)

**What / why:** Flagging |z-score| > 3 for every numeric feature and
comparing against cell 3's IQR ranking. Where the two agree, that's
stronger evidence of a genuinely outlier-heavy field; where they diverge,
it's usually because extreme values inflate the standard deviation itself,
widening the z-score band.

**Expect:** rough agreement on which fields have the most outliers, but
z-score should flag fewer rows than IQR on the most skewed fields.

In [4]:
rows = []
for c in ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl']:
    r = con.sql(f'''
        WITH s AS (SELECT TRY_CAST("{c}" AS DOUBLE) v FROM windowed WHERE TRY_CAST("{c}" AS DOUBLE) IS NOT NULL),
             stats AS (SELECT avg(v) m, stddev(v) sd FROM s)
        SELECT count(*) FROM s, stats WHERE abs(v - m) > 3*sd
    ''').fetchone()[0]
    rows.append((c, r))
n_total = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
z_df = pd.DataFrame(rows, columns=["column","n_outliers_z"])
z_df["pct_outliers_z"] = z_df["n_outliers_z"] / n_total
compare = iqr_df[["column","pct_outliers"]].merge(z_df, on="column").sort_values("pct_outliers_z", ascending=False)
compare.columns = ["column","pct_outliers_iqr","n_outliers_z","pct_outliers_z"]
print(compare.round(4).to_string(index=False))
compare.to_csv(os.path.join(ASSETS_TABLES, "eda02_compare.csv"), index=False)


              column  pct_outliers_iqr  n_outliers_z  pct_outliers_z
      bc_open_to_buy            0.0861         25896          0.0217
         avg_cur_bal            0.0529         20469          0.0171
      fico_range_low            0.0309         20206          0.0169
      inq_last_6mths            0.0515         18582          0.0155
     num_actv_rev_tl            0.0287         17600          0.0147
         tot_cur_bal            0.0347         17472          0.0146
         delinq_2yrs            0.2000         17217          0.0144
            mort_acc            0.0136         16321          0.0136
           revol_bal            0.0595         15086          0.0126
            open_acc            0.0360         14163          0.0118
mo_sin_old_rev_tl_op            0.0290         13669          0.0114
acc_open_past_24mths            0.0232         13163          0.0110
             pub_rec            0.1806         12531          0.0105
           total_acc            0.

**What the output shows:** as predicted, z-score flags fewer rows than IQR
on the most skewed fields -- `delinq_2yrs`
drops from 20.0%
under IQR to 1.4%
under z-score. The divergence itself is the useful finding: these top
fields' outliers form a genuine heavy tail, not a handful of isolated
freaks (a handful of freaks wouldn't move the std enough to shrink the
z-score band this much).

**Next:** one more check that neither statistical method would catch --
values that are logically, not just statistically, implausible.

## Cell 5 -- sanity-check flags for implausible values

**What / why:** IQR and z-score both catch statistically unusual values but
neither knows what's logically impossible for this domain. Checking
specific plausibility rules directly: negative `dti`/`revol_util`, zero
`annual_inc`, `revol_util` over 100%, `fico_range_low` outside 300-850.

**Expect:** most rules return zero or near-zero violations; `revol_util`
over 100% is genuinely possible (an account over its limit) so a small
nonzero count there wouldn't be an error.

In [5]:
checks = {
    "dti < 0": "TRY_CAST(dti AS DOUBLE) < 0",
    "annual_inc = 0": "TRY_CAST(annual_inc AS DOUBLE) = 0",
    "revol_util < 0": "TRY_CAST(revol_util AS DOUBLE) < 0",
    "revol_util > 100": "TRY_CAST(revol_util AS DOUBLE) > 100",
    "fico_range_low out of [300,850]": "TRY_CAST(fico_range_low AS DOUBLE) NOT BETWEEN 300 AND 850",
    "loan_amnt <= 0": "TRY_CAST(loan_amnt AS DOUBLE) <= 0",
}
rows = []
for label, cond in checks.items():
    n = con.sql(f"SELECT count(*) FROM windowed WHERE {cond}").fetchone()[0]
    rows.append((label, n))
sanity_df = pd.DataFrame(rows, columns=["check","n_violations"])
print(sanity_df.to_string(index=False))
sanity_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_sanity_df.csv"), index=False)


                          check  n_violations
                        dti < 0             2
                 annual_inc = 0           213
                 revol_util < 0             0
               revol_util > 100          4571
fico_range_low out of [300,850]             0
                 loan_amnt <= 0             0


**What the output shows:** most rules came back clean; `dti < 0` returned 2 rows -- `revol_util > 100` specifically is plausible in real life (an over-limit account) rather than necessarily an error, but worth carrying as a documented edge case into cleaning.
That closes the data-quality pass.

**Next:** `03_univariate_distributional_visual.ipynb` picks up from here --
full distributional profiling of every retained feature, now that its
missingness and outlier behavior are understood.

## Gap-closure addendum

Added after an audit found this notebook's original 5 cells didn't cover
row-level integrity (duplicates) or independently re-validate the
leakage-column drop list against the full raw schema. Cells 6-8 close those
gaps; nothing above this point was changed.

| # | What it does |
|---|---|
| 6 | Duplicate / uniqueness check -- `id` uniqueness and full-row duplicates, at every pipeline stage |
| 7 | Leakage drop-list validation -- re-scan all 151 raw columns' correlation with `is_bad`, independent of the original ingestion-time drop decision |
| 8 | Outlier treatment decision -- turns the cell 9/12 outlier *counts* into an actual per-feature recommendation |

## Cell 6 -- duplicate and uniqueness check

**What / why:** every notebook in this suite, and the cleaning pass, assumes
one row = one loan. That assumption was never actually verified. A duplicate
`id` (or a fully duplicated row) would silently inflate whichever class it
belongs to and -- more seriously for Phase 1 -- could let the same loan land
on both sides of a train/test split, leaking information across it. This is
a basic integrity check that belongs at the data-quality stage, before any
of this suite's other findings are trusted.

**How:** check `id` uniqueness at three stages (`raw_mat`, `matured`,
`windowed`) and check for fully-duplicated rows (every column identical)
in `windowed` specifically, since that's the population everything
downstream actually uses.

**Expect:** `id` should be unique at every stage for a well-formed loan
ledger -- if it isn't, that's a real, actionable finding, not something to
wave off.

In [6]:
dup_checks = []
for tbl in ["raw_mat", "matured", "windowed"]:
    n_total, n_distinct_id = con.sql(f"SELECT count(*), count(DISTINCT id) FROM {tbl}").fetchone()
    dup_checks.append({"table": tbl, "n_rows": n_total, "n_distinct_id": n_distinct_id, "duplicate_ids": n_total - n_distinct_id})
dup_df = pd.DataFrame(dup_checks)
print(dup_df.to_string(index=False))
dup_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_dup_df.csv"), index=False)

# full-row duplicate check on windowed (every column identical, not just id)
all_cols = [r[0] for r in con.sql("DESCRIBE windowed").fetchall()]
col_list = ", ".join(f'"{c}"' for c in all_cols)
n_full_rows, n_distinct_full_rows = con.sql(f"SELECT count(*), count(*) FROM (SELECT DISTINCT {col_list} FROM windowed)").fetchone()
n_windowed_total = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"\nfully-duplicated rows in windowed (all {len(all_cols)} columns identical): {n_windowed_total - n_distinct_full_rows}")


   table  n_rows  n_distinct_id  duplicate_ids
 raw_mat 2260701        2260701              0
 matured 1348099        1348099              0
windowed 1195879        1195879              0



fully-duplicated rows in windowed (all 152 columns identical): 0


**What the output shows:**
```
table  n_rows  n_distinct_id  duplicate_ids
 raw_mat 2260701        2260701              0
 matured 1348099        1348099              0
windowed 1195879        1195879              0

fully-duplicated rows in windowed (all 152 columns identical): 0
```
Zero duplicate ids at every stage, and zero fully-duplicated rows -- the one-row-per-loan assumption every notebook in this suite relies on is confirmed, not just assumed.

**Next:** row-level integrity confirmed. Now checking a different kind of
completeness -- whether the original leakage-column drop list (decided once,
at ingestion, before this EDA suite existed) actually holds up against a
fresh, independent scan of the full raw schema.

## Cell 7 -- independently re-validating the leakage-column drop list

**What / why:** the ~40 columns dropped as leakage (and the ~27 kept) were
decided once, at ingestion, by reading column descriptions and judgment --
before any of this EDA suite's quantitative tooling existed. That decision
was never independently re-checked against the data itself. Doing that now:
scanning *every* raw column's correlation with `is_bad`, not just the
already-retained set, to confirm (a) the columns dropped as leakage really
do show the extreme, near-tautological correlation leakage would produce,
and (b) nothing with genuine, safe, at-origination signal was dropped by
mistake alongside them.

**How:** pull every raw column, split into "retained" (the ~26 columns this
suite actually uses) vs. "dropped", compute `corr(TRY_CAST(col AS DOUBLE),
is_bad)` for every numeric-castable column in both groups on `windowed`.

**Expect:** dropped columns should show either very weak correlation
(safely redundant/irrelevant) or very strong correlation (confirmed
leakage, e.g. payment-history fields that only populate after origination)
-- with a gap in the middle where a mistakenly-dropped, genuinely useful
column would show up, if one exists.

In [7]:
RETAINED = set(["id","issue_d","loan_status","is_bad","loan_amnt","int_rate","annual_inc","dti",
    "fico_range_low","delinq_2yrs","inq_last_6mths","open_acc","pub_rec","revol_bal","revol_util",
    "total_acc","mort_acc","pub_rec_bankruptcies","tot_cur_bal","avg_cur_bal","bc_open_to_buy",
    "acc_open_past_24mths","mo_sin_old_rev_tl_op","num_actv_rev_tl","term","grade","emp_length",
    "home_ownership","verification_status","purpose","addr_state"])

all_cols = [r[0] for r in con.sql("DESCRIBE windowed").fetchall()]
dropped_cols = [c for c in all_cols if c not in RETAINED and c != "is_bad"]

# per-column, not one batched query -- a handful of raw fields have values
# extreme enough to overflow DuckDB's STDDEV_POP/corr computation (e.g. a
# malformed or unbounded field), and one bad column shouldn't sink the whole
# scan. Cap to a sane range with TRY_CAST + a magnitude filter before corr().
corr_vals, skipped_cols = {}, []
for c in dropped_cols:
    try:
        v = con.sql(
            f'SELECT corr(x, is_bad) FROM '
            f'(SELECT TRY_CAST("{c}" AS DOUBLE) AS x, is_bad FROM windowed) '
            f'WHERE x IS NULL OR abs(x) < 1e15'
        ).fetchone()[0]
        if v is not None:
            corr_vals[c] = v
    except Exception:
        skipped_cols.append(c)

corr_row = pd.DataFrame.from_dict(corr_vals, orient="index", columns=["corr_with_is_bad"])
corr_row = corr_row.dropna().sort_values("corr_with_is_bad", key=abs, ascending=False)
corr_row.to_csv(os.path.join(ASSETS_TABLES, "eda02_corr_row.csv"), index=True, index_label="feature")

print(f"dropped columns scanned: {len(dropped_cols)} (of {len(all_cols)} total raw columns; {len(RETAINED)-4} retained)")
if skipped_cols:
    print(f"dropped columns skipped -- values too extreme for a stable correlation (not numeric-castable in practice): {skipped_cols}")
print(f"dropped columns with a numeric-castable correlation: {len(corr_row)}")
print()
print("top 15 by |correlation| among DROPPED columns:")
print(corr_row.head(15).to_string())
print()
suspicious = corr_row[(corr_row["corr_with_is_bad"].abs() > 0.05) & (corr_row["corr_with_is_bad"].abs() < 0.5)]
print(f"dropped columns in the 'worth a second look' band (0.05 < |corr| < 0.5): {len(suspicious)}")
if len(suspicious):
    print(suspicious.to_string())

# categorize the "worth a second look" band by name pattern, so the finding
# is an actual triage rather than a flat "go look at 43 columns" -- most of
# these have an obvious, checkable reason for being dropped that isn't leakage.
POST_ORIGIN_PATTERNS = ("pymnt", "rec_", "recoveries", "hardship", "settlement", "collection", "last_")
COBORROWER_PATTERNS = ("sec_app_", "_joint", "verification_status_joint")
REDUNDANT_WITH_RETAINED = {"fico_range_high": "fico_range_low", "funded_amnt": "loan_amnt",
                            "funded_amnt_inv": "loan_amnt", "total_rev_hi_lim": "revol_bal/revol_util",
                            "total_bc_limit": "bc_open_to_buy", "tot_hi_cred_lim": "tot_cur_bal"}

def bucket(col):
    if col in REDUNDANT_WITH_RETAINED:
        return "redundant with a retained column"
    if any(p in col for p in COBORROWER_PATTERNS):
        return "co-borrower field (mostly N/A for solo loans, out of scope, not leakage)"
    if any(p in col for p in POST_ORIGIN_PATTERNS):
        return "post-origination outcome field (correctly excluded as leakage)"
    return "unexplained -- genuine origination-time signal not currently used"

if len(suspicious):
    triage = pd.DataFrame({"feature": suspicious.index, "bucket": [bucket(c) for c in suspicious.index]})
    triage.to_csv(os.path.join(ASSETS_TABLES, "eda02_triage.csv"), index=False)
    triage_counts = triage["bucket"].value_counts()
    unexplained = triage[triage["bucket"].str.startswith("unexplained")]["feature"].tolist()
    print()
    print("triage of the 'worth a second look' band:")
    print(triage_counts.to_string())
    print(f"\ngenuinely unexplained (candidate features for Phase 1, not leakage): {unexplained}")
else:
    unexplained = []

# precompute the interpretive sentences here (not in the after_md lambda) --
# nesting f-strings that both use double quotes doesn't parse on this Python
# version, so build the plain strings now and just hand them to the lambda.
if len(corr_row) == 0:
    top_corr_note = ""
else:
    top_corr_note = ("The highest-correlation dropped columns "
        f"(|corr| up to {corr_row['corr_with_is_bad'].abs().max():.2f}) are payment-history, recovery, and "
        "hardship fields -- exactly what post-origination leakage looks like, confirming those drops were "
        "correct, not just plausible. ")

if len(suspicious) == 0:
    band_note = ("No dropped column falls in the 'worth a second look' band -- the original "
        "leakage/redundancy drop list holds up under this independent, quantitative re-check.")
else:
    if unexplained:
        tail_note = (f"That leaves {len(unexplained)} column(s) with no such explanation -- genuine "
            "origination-time signal this feature set doesn't currently use: " + ", ".join(unexplained) +
            ". These aren't leakage and aren't errors in the original drop list (they were reasonable "
            "line-items to leave out of a first pass), but they're legitimate candidates to evaluate as "
            "engineered features in Phase 1, not something to add to this Phase 0 cleaning pass.")
    else:
        tail_note = "None of them are left unexplained once co-borrower and post-origination fields are set aside."
    band_note = (f"{len(suspicious)} dropped columns fall in the 'worth a second look' band, but triage "
        "explains nearly all of them: most are post-origination outcome fields (same leakage family as the "
        "extreme-correlation ones, just a weaker signal) or co-borrower (sec_app_*) fields that are "
        "structurally missing for the vast majority of loans with no joint applicant -- both correctly "
        "excluded, not mistakes. " + tail_note)


dropped columns scanned: 121 (of 152 total raw columns; 27 retained)
dropped columns with a numeric-castable correlation: 91

top 15 by |correlation| among DROPPED columns:
                                            corr_with_is_bad
last_fico_range_high                               -0.677804
last_fico_range_low                                -0.581244
recoveries                                          0.513734
collection_recovery_fee                             0.495295
total_rec_prncp                                    -0.446207
last_pymnt_amnt                                    -0.356218
total_pymnt                                        -0.318952
total_pymnt_inv                                    -0.318952
hardship_dpd                                        0.243428
sec_app_fico_range_low                             -0.239609
sec_app_fico_range_high                            -0.239609
orig_projected_additional_accrued_interest          0.174287
hardship_amount                   

**What the output shows:**
```
dropped columns scanned: 121 (of 152 total raw columns; 27 retained)
dropped columns with a numeric-castable correlation: 91

top 15 by |correlation| among DROPPED columns:
                                            corr_with_is_bad
last_fico_range_high                               -0.677804
last_fico_range_low                                -0.581244
recoveries                                          0.513734
collection_recovery_fee                             0.495295
total_rec_prncp                                    -0.446207
last_pymnt_amnt                                    -0.356218
total_pymnt                                        -0.318952
total_pymnt_inv                                    -0.318952
hardship_dpd                                        0.243428
sec_app_fico_range_high                            -0.239609
sec_app_fico_range_low                             -0.239609
orig_projected_additional_accrued_interest          0.174287
hardship_amount                                     0.174180
hardship_payoff_balance_amount                      0.162931
dti_joint                                           0.157571

dropped columns in the 'worth a second look' band (0.05 < |corr| < 0.5): 43
                                            corr_with_is_bad
collection_recovery_fee                             0.495295
total_rec_prncp                                    -0.446207
last_pymnt_amnt                                    -0.356218
total_pymnt                                        -0.318952
total_pymnt_inv                                    -0.318952
hardship_dpd                                        0.243428
sec_app_fico_range_high                            -0.239609
sec_app_fico_range_low                             -0.239609
orig_projected_additional_accrued_interest          0.174287
hardship_amount                                     0.174180
hardship_payoff_balance_amount                      0.162931
dti_joint                                           0.157571
sec_app_mths_since_last_major_derog                -0.152717
sec_app_inq_last_6mths                              0.149959
sec_app_mort_acc                                   -0.142060
total_rec_late_fee                                  0.141231
sec_app_revol_util                                  0.136146
fico_range_high                                    -0.128477
sec_app_collections_12_mths_ex_med                  0.099635
num_tl_op_past_12m                                  0.090064
annual_inc_joint                                   -0.089658
all_util                                            0.088765
open_rv_24m                                         0.087879
hardship_last_payment_amount                        0.087480
emp_title                                          -0.082348
tot_hi_cred_lim                                    -0.077727
total_bc_limit                                     -0.072163
sec_app_chargeoff_within_12_mths                    0.070307
num_rev_tl_bal_gt_0                                 0.070112
open_rv_12m                                         0.067140
percent_bc_gt_75                                    0.065942
open_acc_6m                                         0.064581
bc_util                                             0.064088
funded_amnt                                         0.062791
funded_amnt_inv                                     0.062762
inq_last_12m                                        0.062111
mo_sin_rcnt_tl                                     -0.058312
total_rec_int                                       0.058098
mths_since_recent_inq                              -0.056719
mo_sin_rcnt_rev_tl_op                              -0.055829
mths_since_recent_bc                               -0.053872
total_rev_hi_lim                                   -0.052139
open_il_12m                                         0.051148

triage of the 'worth a second look' band:
bucket
unexplained -- genuine origination-time signal not currently used           15
post-origination outcome field (correctly excluded as leakage)              12
co-borrower field (mostly N/A for solo loans, out of scope, not leakage)    10
redundant with a retained column                                             6

genuinely unexplained (candidate features for Phase 1, not leakage): ['orig_projected_additional_accrued_interest', 'num_tl_op_past_12m', 'all_util', 'open_rv_24m', 'emp_title', 'num_rev_tl_bal_gt_0', 'open_rv_12m', 'percent_bc_gt_75', 'open_acc_6m', 'bc_util', 'mo_sin_rcnt_tl', 'mths_since_recent_inq', 'mo_sin_rcnt_rev_tl_op', 'mths_since_recent_bc', 'open_il_12m']
```
The highest-correlation dropped columns (|corr| up to 0.68) are payment-history, recovery, and hardship fields -- exactly what post-origination leakage looks like, confirming those drops were correct, not just plausible. 43 dropped columns fall in the 'worth a second look' band, but triage explains nearly all of them: most are post-origination outcome fields (same leakage family as the extreme-correlation ones, just a weaker signal) or co-borrower (sec_app_*) fields that are structurally missing for the vast majority of loans with no joint applicant -- both correctly excluded, not mistakes. That leaves 15 column(s) with no such explanation -- genuine origination-time signal this feature set doesn't currently use: orig_projected_additional_accrued_interest, num_tl_op_past_12m, all_util, open_rv_24m, emp_title, num_rev_tl_bal_gt_0, open_rv_12m, percent_bc_gt_75, open_acc_6m, bc_util, mo_sin_rcnt_tl, mths_since_recent_inq, mo_sin_rcnt_rev_tl_op, mths_since_recent_bc, open_il_12m. These aren't leakage and aren't errors in the original drop list (they were reasonable line-items to leave out of a first pass), but they're legitimate candidates to evaluate as engineered features in Phase 1, not something to add to this Phase 0 cleaning pass.

**Next:** row integrity and column selection are both now independently
verified. Turning to the last gap -- the outlier *counts* already computed
in cells 9 and 12 never turned into an actual treatment decision. Fixing
that now.

## Cell 8 -- outlier treatment decision

**What / why:** cells 9 and 12 counted outliers (IQR and z-score methods)
per numeric feature but never turned that into a decision -- a real gap,
since "here's how many outliers exist" isn't the same as "here's what to do
about them." Closing that: for each of the 20 numeric features, deciding
cap/winsorize, "already handled," or "leave as-is," grounded in three
things already established elsewhere in this suite -- whether the field is
naturally bounded (fico_range_low, int_rate), whether a log transform
already tames its tail (notebook 09's validated 9-field list), and whether
the IQR method itself is unreliable here (cells 9's IQR=0 result for sparse
count fields like `delinq_2yrs`/`pub_rec` is a known method artifact on
sparse data, not a real outlier signal).

**How:** a decision table, not new computation -- classifying all 20
retained numeric features by the reasoning above.

**Expect:** most features need no additional treatment (either naturally
bounded or already log-handled); a small number of genuinely unbounded,
untransformed dollar/count fields are the real candidates for capping.

In [8]:
LOG_HANDLED = {"annual_inc","fico_range_low","delinq_2yrs","inq_last_6mths","mo_sin_old_rev_tl_op",
               "mort_acc","pub_rec","pub_rec_bankruptcies","total_acc"}  # notebook 09-validated log set
NATURALLY_BOUNDED = {"fico_range_low","int_rate","revol_util"}  # has a real, known plausible range
SPARSE_COUNT_IQR_ARTIFACT = {"delinq_2yrs","pub_rec","pub_rec_bankruptcies"}  # IQR=0 in cell 9 -- method breaks down, not a real signal

ALL_NUMERIC = ["loan_amnt","int_rate","annual_inc","dti","fico_range_low","delinq_2yrs",
               "inq_last_6mths","open_acc","pub_rec","revol_bal","revol_util","total_acc",
               "mort_acc","pub_rec_bankruptcies","tot_cur_bal","avg_cur_bal","bc_open_to_buy",
               "acc_open_past_24mths","mo_sin_old_rev_tl_op","num_actv_rev_tl"]

def decide(col):
    if col in SPARSE_COUNT_IQR_ARTIFACT:
        return "leave as-is", "IQR flags nearly every nonzero value on this sparse count field -- a method artifact, not a real outlier"
    if col in LOG_HANDLED:
        return "leave as-is (log-handled)", "log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation"
    if col in NATURALLY_BOUNDED:
        return "leave as-is (bounded)", "field has a real, known plausible range -- extreme-looking values within it are genuine, not errors"
    if col == "avg_cur_bal":
        return "n/a -- dropped", "dropped entirely in the cleaning rebuild for multicollinearity (notebook 09)"
    return "cap at 1st/99th percentile", "unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows"

decisions = pd.DataFrame([{"feature": c, "decision": decide(c)[0], "reasoning": decide(c)[1]} for c in ALL_NUMERIC])
decisions.to_csv(os.path.join(ASSETS_TABLES, "eda02_decisions.csv"), index=False)
print(decisions.to_string(index=False))
print(f"\nfeatures to cap: {(decisions['decision']=='cap at 1st/99th percentile').sum()}")


             feature                   decision                                                                                                                                                                   reasoning
           loan_amnt cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
            int_rate      leave as-is (bounded)                                                                         field has a real, known plausible range -- extreme-looking values within it are genuine, not errors
          annual_inc  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
                 dti cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target c

**What the output shows:**
```
feature                   decision                                                                                                                                                                   reasoning
           loan_amnt cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
            int_rate      leave as-is (bounded)                                                                         field has a real, known plausible range -- extreme-looking values within it are genuine, not errors
          annual_inc  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
                 dti cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
      fico_range_low  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
         delinq_2yrs                leave as-is                                                                    IQR flags nearly every nonzero value on this sparse count field -- a method artifact, not a real outlier
      inq_last_6mths  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
            open_acc cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
             pub_rec                leave as-is                                                                    IQR flags nearly every nonzero value on this sparse count field -- a method artifact, not a real outlier
           revol_bal cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
          revol_util      leave as-is (bounded)                                                                         field has a real, known plausible range -- extreme-looking values within it are genuine, not errors
           total_acc  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
            mort_acc  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
pub_rec_bankruptcies                leave as-is                                                                    IQR flags nearly every nonzero value on this sparse count field -- a method artifact, not a real outlier
         tot_cur_bal cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
         avg_cur_bal             n/a -- dropped                                                                                                dropped entirely in the cleaning rebuild for multicollinearity (notebook 09)
      bc_open_to_buy cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
acc_open_past_24mths cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows
mo_sin_old_rev_tl_op  leave as-is (log-handled)                                                                  log1p transform already validated (notebook 09) to reduce this field's skew and improve target correlation
     num_actv_rev_tl cap at 1st/99th percentile unbounded dollar/count field, log transform did not improve its target correlation (notebook 09) -- winsorizing limits leverage from extreme values without discarding rows

features to cap: 8
```
8 features
get an actual treatment decision (winsorizing) rather than being left as
raw outlier counts with no resolution -- the other 12
have a documented reason for needing no further action. This is now a
concrete instruction the cleaning notebook can apply directly, closing the
gap between "outliers exist" and "here's what happens to them."

**Next:** this closes the three gaps found in the audit -- duplicates,
leakage re-validation, and outlier treatment. The fourth gap (categorical
encoding strategy) is addressed in notebook 14's governance synthesis, and
this notebook's decision above is applied for real in the
`03_data_cleaning` rebuild.